# E1 -- double well (1D): run notebook

**This notebook runs and saves. It does not typeset figures.**

Every official metric is computed here, at run time, and written into each run's `metrics_timeseries.csv` and `cost_timeseries.csv`. The companion notebook `E1_double_well_plot.ipynb` reads those numbers and never recomputes them.

**Run All executes the single default full configuration.** There is exactly one configuration for this experiment, `configs/experiments/E1.yaml` -- there is no smoke, dev, reduced, or production profile to choose between. Lowering the particle count for local debugging is an explicit temporary edit, never a second committed profile.

Each variant is saved the moment it finishes, into its own atomically renamed run directory, so a variant that fails leaves the earlier ones untouched.

In [1]:
import sys

sys.path.insert(0, "..")  # importable when launched from notebooks/

from src.pipeline import load_experiment, run_variants_and_save

## Target, reference, and cost calibration

The reference is built **once** and reused by every method. It does not depend on any method parameter, so it is **never rebuilt per method, per hyperparameter value, or per canonical/tamed variant**; a cached reference on disk is loaded instead of being recomputed.

The force-equivalent-evaluation (FEE) calibration is measured once per device in the same way, and every run in this experiment is costed against that one calibration. The device is resolved automatically -- no device index is pinned in this notebook.

In [2]:
experiment = load_experiment("E1", device="auto")

reference = experiment.ensure_reference()
fee = experiment.ensure_fee_calibration()

described = reference.describe()
print(f"reference: kind={described.get('kind', described.get('method'))}  hash={experiment.reference_hash}")
print(f"FEE:       unit={fee.cost_unit}  hash={fee.hash}")

reference: kind=grid_inverse_cdf_1d  hash=5f1aeed3092c00874ea7c25eb08e636d
FEE:       unit=amortized_time_per_configuration  hash=3590adff094714fc599081a4fb309ee3


## ULA

Every taming-capable method runs **both** a canonical and a tamed variant. `run_variants_and_save` expands each entry of `variants` into those two runs by itself, so a notebook never passes `tame`. The two variants are **calibrated separately** -- each one gets its own step size from its own `dt` refinement -- and each is saved as its own run directory.

In [3]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULA",
    # `run_variants_and_save` expands each entry below into a
    # canonical and a tamed run, so `tame` is never passed here.
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[ULA, canonical] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/ULA/ULA-canonical-dt0.005-20260806T212405509114Z


[ULA, tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/ULA/ULA-tamed-dt0.005-20260806T212412975526Z


[{'variant_label': 'ULA, canonical',
  'status': 'complete',
  'run_id': 'ULA-canonical-dt0.005-20260806T212405509114Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/ULA/ULA-canonical-dt0.005-20260806T212405509114Z',
  'dt': 0.005,
  'calibration_hash': '07f693b6f5028073efca116c9a323a77',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'ULA, tamed',
  'status': 'complete',
  'run_id': 'ULA-tamed-dt0.005-20260806T212412975526Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/ULA/ULA-tamed-dt0.005-20260806T212412975526Z',
  'dt': 0.005,
  'calibration_hash': '689faa00e8bda7659f8c2515e330a93a',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## MALA

MALA supports taming, so it also runs both variants. Tamed MALA implements the actual tamed proposal density in the Metropolis-Hastings ratio; it is a genuine second sampler, not a relabelled copy of the canonical run.

In [4]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="MALA",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[MALA, canonical] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/MALA/MALA-canonical-dt0.16-20260806T212417129953Z


[MALA, tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/MALA/MALA-tamed-dt0.16-20260806T212421353606Z


[{'variant_label': 'MALA, canonical',
  'status': 'complete',
  'run_id': 'MALA-canonical-dt0.16-20260806T212417129953Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/MALA/MALA-canonical-dt0.16-20260806T212417129953Z',
  'dt': 0.16,
  'calibration_hash': '5be7020120df7dd4f7221539976c6940',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 628,
  'n_snapshots': 4},
 {'variant_label': 'MALA, tamed',
  'status': 'complete',
  'run_id': 'MALA-tamed-dt0.16-20260806T212421353606Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/MALA/MALA-tamed-dt0.16-20260806T212421353606Z',
  'dt': 0.16,
  'calibration_hash': '2ad6c681a51af7e34f4566116b026712',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 628,
  'n_snapshots': 4}]

## FLA

The three stability indices are this experiment's default grid in `configs/registry.yaml`. All three run from this one cell and save as separate variants, and each of them is expanded into a canonical and a tamed run.

In [5]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="FLA",
    variants=[
        {"alpha": 1.6}, {"alpha": 1.7}, {"alpha": 1.8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[FLA alpha=1.6, canonical] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/FLA/FLA-alpha1.6-canonical-dt0.0025-20260806T212433277817Z


[FLA alpha=1.6, tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/FLA/FLA-alpha1.6-tamed-dt0.005-20260806T212442427890Z


[FLA alpha=1.7, canonical] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/FLA/FLA-alpha1.7-canonical-dt0.0025-20260806T212455414491Z


[FLA alpha=1.7, tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/FLA/FLA-alpha1.7-tamed-dt0.005-20260806T212504526770Z


[FLA alpha=1.8, canonical] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/FLA/FLA-alpha1.8-canonical-dt0.0025-20260806T212516474798Z


[FLA alpha=1.8, tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/FLA/FLA-alpha1.8-tamed-dt0.005-20260806T212525615638Z


[{'variant_label': 'FLA alpha=1.6, canonical',
  'status': 'complete',
  'run_id': 'FLA-alpha1.6-canonical-dt0.0025-20260806T212433277817Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/FLA/FLA-alpha1.6-canonical-dt0.0025-20260806T212433277817Z',
  'dt': 0.0025,
  'calibration_hash': 'c557be8a4afae9e1409d96497cf04fe7',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'FLA alpha=1.6, tamed',
  'status': 'complete',
  'run_id': 'FLA-alpha1.6-tamed-dt0.005-20260806T212442427890Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/FLA/FLA-alpha1.6-tamed-dt0.005-20260806T212442427890Z',
  'dt': 0.005,
  'calibration_hash': '49ed2c3bc143a01b02be5b636d6ce8e3',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'FLA alpha=1.7, canonical',
  'status': 'complete',
  'run_id': 'FLA-alp

## ULD

ULD is the method; BAOAB is the integrator it is discretised with. Runs, manifests, and legends say ULD.

In [6]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULD",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[ULD gamma=1, canonical] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/ULD/ULD-gamma1-canonical-dt0.005-20260806T212532835055Z


[ULD gamma=1, tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/ULD/ULD-gamma1-tamed-dt0.005-20260806T212540618007Z


[{'variant_label': 'ULD gamma=1, canonical',
  'status': 'complete',
  'run_id': 'ULD-gamma1-canonical-dt0.005-20260806T212532835055Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/ULD/ULD-gamma1-canonical-dt0.005-20260806T212532835055Z',
  'dt': 0.005,
  'calibration_hash': '9bec57e8432e3a5294a7a0f2037c0c53',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'ULD gamma=1, tamed',
  'status': 'complete',
  'run_id': 'ULD-gamma1-tamed-dt0.005-20260806T212540618007Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/ULD/ULD-gamma1-tamed-dt0.005-20260806T212540618007Z',
  'dt': 0.005,
  'calibration_hash': '8c2dc1c2fa7c63be45f6b7b7bb4415a6',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## PT

Parallel tempering. The replica ladder is tuned by the calibration step that `run_variants_and_save` invokes, not here, and the tuned ladder is written into the run's `calibration.json`.

In [7]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="PT",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[PT n_swap=10, canonical] NOT CALIBRATABLE: no timestep agreed with its halving on summary_iqr, summary_mean


[PT n_swap=10, tamed] NOT CALIBRATABLE: no timestep agreed with its halving on summary_iqr, summary_mean


[{'variant_label': 'PT n_swap=10, canonical',
  'method': 'PT',
  'status': 'uncalibratable',
  'diagnosis': 'no timestep agreed with its halving on summary_iqr, summary_mean',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.16,
    'pass': False,
    'stability_problems': [],
    'agreement_failures': [{'key': 'summary_mean',
      'coarse': -0.6788742,
      'fine': -0.3589444,
      'difference': 0.3199298,
      'allowance': 0.20434814},
     {'key': 'summary_iqr',
      'coarse': 0.24642013,
      'fine': 1.90202282,
      'difference': 1.6556027,
      'allowance': 0.17913521}],
    'summary': {'n_steps': 39,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.0,
     'n_effective': 1024,
     'summary_mean': -0.6788741980764612,
     'summary_mean_se': 0.02248985247220958,
     'summary_abs_mean': 0.9751246674839495,
     'summary_abs_mean_se': 0.004206977395265193,
     'summary_median': -0.959085978689508,
     'summary_median_se': 0.008437827

## Raw-CP

The same compound-Poisson jump process with the Levy score correction switched off. It does not preserve the target, so it is the control arm that isolates what the score correction buys, not a competitive baseline.

In [8]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="Raw-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[Raw-CP, canonical] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/Raw-CP/Raw-CP-canonical-dt0.005-20260806T212947116719Z


[Raw-CP, tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/Raw-CP/Raw-CP-tamed-dt0.005-20260806T212958825854Z


[{'variant_label': 'Raw-CP, canonical',
  'status': 'complete',
  'run_id': 'Raw-CP-canonical-dt0.005-20260806T212947116719Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/Raw-CP/Raw-CP-canonical-dt0.005-20260806T212947116719Z',
  'dt': 0.005,
  'calibration_hash': 'f6acd46827047d5565311948546fa8ce',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'Raw-CP, tamed',
  'status': 'complete',
  'run_id': 'Raw-CP-tamed-dt0.005-20260806T212958825854Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E1_double_well/runs/Raw-CP/Raw-CP-tamed-dt0.005-20260806T212958825854Z',
  'dt': 0.005,
  'calibration_hash': '31b4f72be87b6c68f39227acc78ba117',
  'fee_calibration_hash': '3590adff094714fc599081a4fb309ee3',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## LSC-CP

Compound-Poisson jumps with the full deterministic-quadrature Levy score correction.

In [9]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[LSC-CP, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP, tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/LSC-CP/LSC-CP-tamed-dt0.005-20260806T213100498757Z


[{'variant_label': 'LSC-CP, canonical',
  'method': 'LSC-CP',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.005,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.7076484375)],
    'agreement_failures': [],
    'summary': {'n_steps': 1250,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.7076484375,
     'n_effective': 1024,
     'summary_mean': -0.9942070990820988,
     'summary_mean_se': 0.08055583674284288,
     'summary_abs_mean': 2.857250433399642,
     'summary_abs_mean_se': 0.013741954496533136,
     'summary_median': -2.819816629035433,
     'summary_median_se': 0.014660204288690996,
     'summary_iqr': 5.797622445052054,
     'summary_iqr_se': 0.037088315929721165,
     'energy_mean': 58.850864495954845,
     'energy_mean_se': 0.7178927364470441,
     'energy

## LSC-CP-RA

`A` is the **iid Monte Carlo bank size of one estimator family**, LSC-CP-RA. `A = 1, 4, 8` are variants of that single family, not three separate methods, and **all of them run from this one cell** and save as separate variants.

The bank holds `A` displacements drawn iid from the full normalised jump law `rho = nu / lambda`, and **the same bank drives both the score and the compound-Poisson increment**.

In [10]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP-RA",
    variants=[
        {"A": 1}, {"A": 4}, {"A": 8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[LSC-CP-RA, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA, tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/LSC-CP-RA/LSC-CP-RA-A1-tamed-dt0.005-20260806T213155335596Z


[LSC-CP-RA (A=4), canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA (A=4), tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/LSC-CP-RA/LSC-CP-RA-A4-tamed-dt0.005-20260806T213251466645Z


[LSC-CP-RA (A=8), canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA (A=8), tamed] saved to /home/zheyuanlai/levy-sampling/results/E1_double_well/runs/LSC-CP-RA/LSC-CP-RA-A8-tamed-dt0.005-20260806T213347678102Z


[{'variant_label': 'LSC-CP-RA, canonical',
  'method': 'LSC-CP-RA',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.005,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.0492625)],
    'agreement_failures': [],
    'summary': {'n_steps': 1250,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.0492625,
     'n_effective': 1024,
     'summary_mean': 0.019629833945555168,
     'summary_mean_se': 0.03584275539650207,
     'summary_abs_mean': 1.0504504527510634,
     'summary_abs_mean_se': 0.010398761719020754,
     'summary_median': 0.6242640366237888,
     'summary_median_se': 0.663621913924238,
     'summary_iqr': 1.9670892669291176,
     'summary_iqr_se': 0.011323725865379781,
     'energy_mean': 1.0171273487113783,
     'energy_mean_se': 0.15710847746506695,
     'energ

## Rebuild the catalog

`catalog.csv` is a **derived index** over the run manifests. It is never written by a worker mid-run, so concurrent runs never contend for it, and it can be rebuilt at any time by rescanning the manifests -- a lost or stale catalog costs nothing. Only runs that verify (manifest present, `COMPLETE` present, hashes matching) are admitted.

In [11]:
from src.catalog import write_catalog

report = write_catalog(experiment.paths.experiment_dir)
print(f"catalog rebuilt: {report['n_runs']} runs, {report['n_rejected']} rejected")

catalog rebuilt: 72 runs, 34 rejected
